# Laboratorio: Modelos de regresión y clasificación

**Duración estimada:** 60–90 minutos

En este laboratorio seguiremos un proceso de ML: **explorar brevemente**, **elegir modelos razonados**, **entrenar** y **evaluar**. Trabajamos con:
- **Clasificación (Wine)**: 13 atributos químicos de vinos, 3 clases.
- **Regresión (Diabetes)**: progresión de la enfermedad.

## Reglas
- Completa **solo** en `#your code here`.
- No cambies nombres de variables/funciones solicitadas ni el orden de retornos.
- Usa solo `numpy`, `pandas`, `matplotlib`, `scikit-learn`.
- Cada pregunta tiene *sanity checks* visibles y **tests ocultos**

In [ ]:
# === Setup (no modificar) ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.precision', 3)

from sklearn.datasets import load_wine, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Setup OK')
import sklearn, sys
print('Python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__)

## Parte A — Clasificación con Wine
### A.1 Exploración breve
Observa estadísticas y relaciones generales para entender escalas y correlaciones antes de entrenar.

In [ ]:
# A.1 — EDA
wine = load_wine()
Xw = pd.DataFrame(wine.data, columns=wine.feature_names)
yw = pd.Series(wine.target, name='target')
print('Shape:', Xw.shape)
display(Xw.head())
display(Xw.describe())

# Mapa de correlación simplificado
corr = Xw.corr(numeric_only=True)
plt.figure(figsize=(6,5))
plt.imshow(corr, interpolation='nearest')
plt.title('Correlaciones Wine')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(corr.columns)), corr.columns, fontsize=7)
plt.colorbar(fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

### ¿Qué modelos usaremos y por qué?
- **LogisticRegression**: baseline lineal multiclase, interpretable, efectivo si clases separables tras **escalamiento**.
- **KNeighborsClassifier (k=5)**: no paramétrico, útil cuando fronteras pueden ser no lineales; sensible a la escala → **recomendado escalar**.

### A.2 Tarea 1 — División y  escalamiento (10 pts)
Implementa `split_scale_wine(scale=True)` que retorne `(X_train, X_test, y_train, y_test)` usando `test_size=0.2`, `random_state=RANDOM_STATE`, estratificando por `yw`. Si `scale=True`, aplica `StandardScaler` ajustado en *train*.

In [ ]:
def split_scale_wine(scale=True):
    """Divide Xw, yw; si scale=True aplica StandardScaler (ajustado en train).
    Retorna X_train, X_test, y_train, y_test en ese orden"""
    # your code here
    raise NotImplementedError


In [ ]:
# Sanity check visible
Xa_tr, Xa_te, ya_tr, ya_te = split_scale_wine(scale=True)
assert Xa_tr.shape[0] + Xa_te.shape[0] == Xw.shape[0]
assert ya_tr.shape[0] + ya_te.shape[0] == yw.shape[0]
print('OK split/scale Wine')

### A.3 Tarea 2 — LogisticRegression: entrenar y predecir (10 pts)
Implementa `fit_predict_logreg_wine(X_train, y_train, X_test)` que retorne `y_pred` usando `LogisticRegression(max_iter=200, random_state=RANDOM_STATE)`.

In [ ]:
def fit_predict_logreg_wine(X_train, y_train, X_test):
    """"Recibe X_train, y_train, X_test. 
    1. crea LogisticRegression(max_iter, random_state)
    2. ajusta a X_train, y_train
    3. predice sobre X_test
    retorna la predicción"""
    # your code here
    raise NotImplementedError

In [ ]:
# Sanity check
yp_log = fit_predict_logreg_wine(Xa_tr, ya_tr, Xa_te)
assert yp_log.shape == ya_te.shape
print('OK logreg predict')


### A.4 Tarea 3 — KNN (k=5): entrenar y predecir (10pts)
Implementa `fit_predict_knn_wine(X_train, y_train, X_test, k=5)` que retorne `y_pred`.

In [ ]:
def fit_predict_knn_wine(X_train, y_train, X_test, k=5):
    # your code here
    raise NotImplementedError

In [ ]:
# Sanity check
yp_knn = fit_predict_knn_wine(Xa_tr, ya_tr, Xa_te, k=5)
assert yp_knn.shape == ya_te.shape
print('OK knn predict')

### A.5 Tarea 4 — Evaluación y matriz de confusión (10pts)
Implementa `evaluate_classification(y_true, y_pred)` que retorne la **tupla** `(accuracy, precision_macro, recall_macro, f1_macro)` y luego grafica la matriz de confusión.

In [ ]:
def evaluate_classification(y_true, y_pred):
    acc, prec, rec, f1 = 0.0, 0.0, 0.0, 0.0
    # your code here
    raise NotImplementedError
    return acc, prec, rec, f1

In [ ]:
# Con este codigo puedes graficar la matriz de confusión (esta parte no es evaluada, solo un sanity check)

for name, yp in [('LogReg', yp_log), ('KNN', yp_knn)]:
    acc, prec, rec, f1 = evaluate_classification(ya_te, yp)
    print(f'{name}: acc={acc:.3f} | prec={prec:.3f} | rec={rec:.3f} | f1={f1:.3f}')
    cm = confusion_matrix(ya_te, yp)
    plt.figure(); plt.imshow(cm, interpolation='nearest'); plt.title(f'Wine - Confusion ({name})')
    plt.xlabel('Predicho'); plt.ylabel('Real')
    for (i, j), v in np.ndenumerate(cm):
        plt.text(j, i, str(v), ha='center', va='center')
    plt.tight_layout(); plt.show()

## Parte B — Regresión con Diabetes
### B.1 Exploración breve
Revisa correlaciones con el objetivo y dispersión para algunas features con mayor |r|.

In [ ]:
# B.1 — EDA
diab = load_diabetes()
Xd = pd.DataFrame(diab.data, columns=diab.feature_names)
yd = pd.Series(diab.target, name='target')
print('Shape:', Xd.shape)
display(Xd.describe())

corr_y = Xd.apply(lambda c: np.corrcoef(c, yd)[0,1])
top4 = corr_y.abs().sort_values(ascending=False).head(4).index.tolist()
print('Top4 correlación con y:', top4)
for col in top4:
    plt.figure(); plt.scatter(Xd[col], yd, s=10)
    plt.title(f'{col} vs target'); plt.xlabel(col); plt.ylabel('target'); plt.tight_layout(); plt.show()

### ¿Qué modelos usaremos y por qué?
- **LinearRegression**: baseline interpretativo; si la relación lineal aproximada domina, rinde bien.
- **KNeighborsRegressor (k=5)**: captura no linealidades simples; necesita escalamiento para distancia significativa.
Y agregamos un **BaselineMean** que predice la media de `y_train` para tener un piso de rendimiento.

### B.2 Tarea 5 — División y escalamiento en Diabetes (10pts)
Implementa `split_scale_diabetes(scale=True)` que retorne `(X_train, X_test, y_train, y_test)` con `test_size=0.2`, `random_state=RANDOM_STATE`. Si `scale=True`, usa `StandardScaler` ajustado en *train*.

In [ ]:
def split_scale_diabetes(scale=True):
    "Retorna (X_train, X_test, y_train, y_test) en ese orden"
    # your code here
    raise NotImplementedError

In [ ]:
# Sanity check
Xb_tr, Xb_te, yb_tr, yb_te = split_scale_diabetes(scale=True)
assert Xb_tr.shape[0] + Xb_te.shape[0] == Xd.shape[0]
print('OK split/scale Diabetes')


### B.3 Tarea 6 — Comparar modelos de regresión
Implementa `compare_regressors_diabetes(X_train, y_train, X_test, y_test)` que devuelva un `DataFrame` con una fila por modelo `['BaselineMean', 'LinearRegression', 'KNeighborsRegressor']` y columnas `['model', 'MSE','MAE','R2']`, ordenado por `MSE` ascendente.

In [ ]:
def compare_regressors_diabetes(X_train, y_train, X_test, y_test):
    # Como ejemplo, dejamos el baseline listo
    rows = []
    # Baseline
    y_hat = np.full_like(y_test, fill_value=np.mean(y_train), dtype=float)
    rows.append(['BaselineMean', mean_squared_error(y_test,y_hat), mean_absolute_error(y_test,y_hat), r2_score(y_test,y_hat)])
    
    #TODO: agregar a rows las dos filas que faltan
    # your code here
    raise NotImplementedError
    df = pd.DataFrame(rows, columns=['model','MSE','MAE','R2']).set_index('model').sort_values('MSE')
    return df


In [ ]:
df_reg = compare_regressors_diabetes(Xb_tr, yb_tr, Xb_te, yb_te)
df_reg

In [ ]:
# Sanity check
assert set(df_reg.columns)=={'MSE','MAE','R2'}
assert {'BaselineMean','LinearRegression','KNeighborsRegressor'}.issubset(df_reg.index)
print('OK metrics DF')


-- fin del laboratorio 04 --